In [1]:
from data_utils import *
import numpy as np
from tqdm import tqdm
import torch
import random
import pickle
import numpy as np
from collections import defaultdict
import scipy.sparse as sp

import os
import random
import pandas as pd
import json
import pickle
import gzip
from tqdm import tqdm
from utils import getBasicScores,getFairnessScores,area_curve_metric,seedSet

/home/sairamv/anaconda3/lib/python3.8/site-packages/setuptools/distutils_patch.py:25: UserWarning: Distutils was imported before Setuptools. This usage is discouraged and may exhibit undesirable behaviors or errors. Please use Setuptools' objects directly or at least import Setuptools first.
  warnings.warn(


In [2]:
DATASET = 'beauty'

In [3]:
DATA_PATH = '../data/'
POS_PATH = 'top-preds/stage-1-POS-only-EXPLS/'
NEG_PATH = 'top-preds/stage-1-NEG-only-EXPLS'
ZERO_PATH = 'top-preds/stage-1-ZERO-EXPLS/'

In [4]:
BATCH_SIZE=100

In [5]:
seed = 300
seedSet(seed)

# Explanation Loading

In [6]:
pos_expls = load_pickle(os.path.join(POS_PATH,f'DEEPFM-{DATASET}-preds.pkl'))

In [7]:
neg_expls = load_pickle(os.path.join(NEG_PATH,f'DEEPFM-{DATASET}-preds.pkl'))

In [8]:
zero_expls = load_pickle(os.path.join(ZERO_PATH,f'DEEPFM-{DATASET}-preds.pkl'))

In [9]:
def map_sorted_preds_to_ranked_items(preds_file):
    ui_scores = preds_file['ui_scores']
    preds = preds_file['preds']
    
    assert isinstance(ui_scores, dict)
    assert isinstance(preds, list) or isinstance(preds, np.ndarray)

    mapped_results = {}
    user_ids = list(ui_scores.keys())
    num_candidates = len(next(iter(ui_scores.values())))  # assume uniform candidate size

    for idx, user_id in tqdm(enumerate(user_ids)):
        user_pred_scores = np.array(preds[idx * num_candidates : (idx + 1) * num_candidates])
        item_rank_dict = ui_scores[user_id]  # {item_id: -rank}
        
        # Invert to get: {-1: item_id1, -2: item_id2, ..., -100: item_id100}
        rank_to_item = {rank: item for item, rank in item_rank_dict.items()}
        
        # Sort predictions in descending order
        sorted_scores = np.sort(user_pred_scores)[::-1]
        
        # Map highest score to -1, second to -2, ..., lowest to -num_candidates
        user_item_score_map = {}
        for i in range(1, num_candidates + 1):
            rank = -i
            item_id = rank_to_item[rank]
            score = sorted_scores[i - 1]
            user_item_score_map[item_id] = (score,rank)
        
        mapped_results[user_id] = user_item_score_map

    return mapped_results

In [10]:
def get_score(u,v,mapping):
    return round(mapping[u][v][0],5),mapping[u][v][1]

In [11]:
def get_top_k(mapping,user, k):
    lst = sorted(mapping[user].items(), key=lambda x:x[1][0],reverse=True)
    if k == -1:
        k = len(lst)
    return [x[0] for x in lst[:k]]

In [12]:
new_pos_expls = map_sorted_preds_to_ranked_items(pos_expls)
new_neg_expls = map_sorted_preds_to_ranked_items(neg_expls)
new_zero_expls = map_sorted_preds_to_ranked_items(zero_expls)

22363it [00:00, 23767.27it/s]
22363it [00:00, 26700.97it/s]
22363it [00:00, 26629.28it/s]


In [13]:
targetItems = readTargetItem(os.path.join(DATA_PATH,DATASET,"targetItems.txt"))
datamaps = load_json(os.path.join(DATA_PATH,DATASET,"datamaps.json"))
targetItems = [int(datamaps["item2id"][x]) for x in targetItems]
targetItems[:5]

[271, 3983, 1979, 7928, 3649]

# Random Popular Item Selection

In [14]:
popItems = [int(x) for x in datamaps['item2id'].values() if int(x) in targetItems]
print(f"Number of popular items: {len(popItems)}")
popItems[:5]

Number of popular items: 1210


[9, 11, 12, 13, 14]

In [15]:
RATIO = 0.01
num_items = int(RATIO * len(datamaps['item2id']))
num_items

121

In [16]:
popItems = np.random.choice(popItems,size=num_items,replace=False)
assert len(popItems) == len(set(popItems))
popItems[:5]

array([ 122, 1708, 7308, 6466, 2083])

# Re-ranking using the algorithm

In [17]:
def rerank_with_pop_demote(user, new_neg_expls, new_zero_expls, popItems):
    """
    Re-rank using zero explanation scores for all items,
    except for popular items where we use the negative explanation scores.

    Parameters:
    - user: the current user to check
    - new_neg_expls: dict {user: {item: (score, -rank)}}
    - new_zero_expls: dict {user: {item: (score, -rank)}}
    - popItems: set of item IDs

    Returns:
    - result: dict {user: {item: score}}
    """
    reranked_scores = {}

    for item in new_zero_expls[user]:
        if item in popItems:
            score,_ = get_score(user, item, new_neg_expls)
            reranked_scores[item] = score
        else:
            score, _ = get_score(user, item, new_zero_expls)
            reranked_scores[item] = score

    return reranked_scores

In [18]:
all_info = []
golds,preds = [],[]
pop_golds = []

ui_scores = dict()
gt = dict()
pop_gt = dict()
top = [1,2,3,5,10,20]

users = list(new_pos_expls.keys())
for stepv, user in tqdm(enumerate(users)):
    user = int(user)
    gold_item = int(zero_expls['gt'][user][0])
    scores_dict = rerank_with_pop_demote(user, new_neg_expls, new_zero_expls, popItems)
    rerank_lst = sorted(scores_dict.items(),key = lambda x:x[1], reverse=True)
    gt[user] = [gold_item]
    pop_gt[user] = list(popItems) # we only check pop item relevance!
    pred_dict = {}

    for j in range(len(rerank_lst)):

        item, score = rerank_lst[j]
        pred_dict[item] = -(j + 1)
        label = int(gold_item == item)
        pop_label = int(item in popItems)
        golds.append(label)
        pop_golds.append(pop_label)
        preds.append(score)
    
    ui_scores[user] = pred_dict

print("# golds: ",len(golds))
print("# pop golds: ",len(pop_golds))
print("# preds: ",len(preds))
print(f"# popular items overall across all users: {sum(pop_golds)}")
print("Original Recommendation Performance")
_, Recommendresults = getBasicScores(ui_scores, gt, top)
print("\nOriginal AUC: ",area_curve_metric(golds,preds)) 
print("Original Fairness Performance")
FairResults = getFairnessScores(ui_scores, targetItems, top, len(datamaps['item2id']))

22363it [00:17, 1267.49it/s]


# golds:  2236300
# pop golds:  2236300
# preds:  2236300
# popular items overall across all users: 22952
Original Recommendation Performance

NDCG@1	Rec@1	Hits@1	Prec@1	MAP@1	MRR@1
0.1583	0.1583	0.1583	0.1583	0.1583	0.1583

NDCG@2	Rec@2	Hits@2	Prec@2	MAP@2	MRR@2
0.1605	0.1617	0.1617	0.0809	0.1600	0.1600

NDCG@3	Rec@3	Hits@3	Prec@3	MAP@3	MRR@3
0.1629	0.1665	0.1665	0.0555	0.1616	0.1616

NDCG@5	Rec@5	Hits@5	Prec@5	MAP@5	MRR@5
0.1683	0.1799	0.1799	0.0360	0.1646	0.1646

NDCG@10	Rec@10	Hits@10	Prec@10	MAP@10	MRR@10
0.1821	0.2234	0.2234	0.0223	0.1701	0.1701

NDCG@20	Rec@20	Hits@20	Prec@20	MAP@20	MRR@20
0.2076	0.3256	0.3256	0.0163	0.1769	0.1769

Original AUC:  0.570560128314312
Original Fairness Performance

PR@1	LTR@1	KLD@1	Gini@1	SDI@1	UHC@1
0.4213	0.5787	0.3503	0.4947	0.4876	0.4213

PR@2	LTR@2	KLD@2	Gini@2	SDI@2	UHC@2
0.3917	0.6083	0.2965	0.5393	0.4766	0.6034

PR@3	LTR@3	KLD@3	Gini@3	SDI@3	UHC@3
0.3767	0.6233	0.2707	0.5565	0.4696	0.7136

PR@5	LTR@5	KLD@5	Gini@5	SDI@5	UHC@5
0.3555	0.6445	0.

In [19]:
print("Negative Explanation Ablation: \nRecommendation Performance wrt Popular Items")
_, Recommendresults = getBasicScores(ui_scores, pop_gt, top)
print("\nPopular Item AUC: ",area_curve_metric(pop_golds,preds))

Negative Explanation Ablation: 
Recommendation Performance wrt Popular Items

NDCG@1	Rec@1	Hits@1	Prec@1	MAP@1	MRR@1
0.0001	0.0000	0.0001	0.0001	0.0001	0.0001

NDCG@2	Rec@2	Hits@2	Prec@2	MAP@2	MRR@2
0.0002	0.0000	0.0002	0.0001	0.0002	0.0002

NDCG@3	Rec@3	Hits@3	Prec@3	MAP@3	MRR@3
0.0003	0.0000	0.0004	0.0001	0.0002	0.0002

NDCG@5	Rec@5	Hits@5	Prec@5	MAP@5	MRR@5
0.0005	0.0000	0.0008	0.0002	0.0003	0.0003

NDCG@10	Rec@10	Hits@10	Prec@10	MAP@10	MRR@10
0.0010	0.0000	0.0026	0.0003	0.0005	0.0005

NDCG@20	Rec@20	Hits@20	Prec@20	MAP@20	MRR@20
0.0044	0.0001	0.0164	0.0009	0.0014	0.0014

Popular Item AUC:  0.3601356308152271


In [20]:
print("Zero Score for all: \nRecommendation Performance wrt Popular Items")
_, Recommendresults = getBasicScores(zero_expls['ui_scores'], pop_gt, top)
print("\nPopular Item AUC: ",area_curve_metric(pop_golds,zero_expls['preds'])) 

Zero Score for all: 
Recommendation Performance wrt Popular Items

NDCG@1	Rec@1	Hits@1	Prec@1	MAP@1	MRR@1
0.0395	0.0003	0.0395	0.0395	0.0395	0.0395

NDCG@2	Rec@2	Hits@2	Prec@2	MAP@2	MRR@2
0.0596	0.0006	0.0713	0.0366	0.0554	0.0554

NDCG@3	Rec@3	Hits@3	Prec@3	MAP@3	MRR@3
0.0746	0.0009	0.1015	0.0354	0.0653	0.0655

NDCG@5	Rec@5	Hits@5	Prec@5	MAP@5	MRR@5
0.0974	0.0014	0.1573	0.0343	0.0774	0.0782

NDCG@10	Rec@10	Hits@10	Prec@10	MAP@10	MRR@10
0.1310	0.0025	0.2628	0.0307	0.0894	0.0920

NDCG@20	Rec@20	Hits@20	Prec@20	MAP@20	MRR@20
0.1616	0.0041	0.3870	0.0248	0.0939	0.1007

Popular Item AUC:  0.5023870812490242


In [21]:
def compute_avg_rank_score(ablation_score_map,popItems,K):
    print(f"K:{K}")
    pop_zero_score = []
    pop_neg_score = []
    pop_zero_rank = []
    pop_neg_rank = []
    pop_diff_score = []
    pop_diff_rank = []
    
    for user in ablation_score_map:
        item_maps = ablation_score_map[user]
        
        item_list = sorted(item_maps.items(),key = lambda x:x[1][0],reverse=True)
        item_list = [x[0] for x in item_list][:K]
        for item in item_list:
            if item in popItems:
                neg_score, neg_rank = item_maps[item]
                zero_score, zero_rank = get_score(user,item, new_zero_expls)
                zero_rank = int(-zero_rank)
                neg_rank = int(-neg_rank)
                pop_zero_score.append(zero_score)
                pop_neg_score.append(neg_score)
                pop_zero_rank.append(zero_rank)
                pop_neg_rank.append(neg_rank)
                
    print("\tlen(pop_neg_score): ",len(pop_neg_score))
    print("\tlen(pop_zero_score): ",len(pop_zero_score))
    print('\n')
    print(f"\tE(pop neg score): {np.mean(pop_neg_score):.6f}")
    print(f"\tE(pop zero score): {np.mean(pop_zero_score):.6f}")
    print(f"\tDiff between E(pop zero score) - E(pop neg score): {np.mean(pop_zero_score) - np.mean(pop_neg_score):.6f}")
    print('\n')
    print(f"\tE(pop neg rank): {np.mean(pop_neg_rank):.6f}")
    print(f"\tE(pop zero rank): {np.mean(pop_zero_rank):.6f}")
    print(f"\tDiff between E(pop_neg_rank) - E(pop_zero_rank): {np.mean(pop_neg_rank) - np.mean(pop_zero_rank):.6f}")

#     DEN = len(ablation_score_map) * K
#     print("\n WITH DEN: ",DEN)
#     print(f"\tE(pop neg score): {(sum(pop_neg_score)/DEN):.6f}")
#     print(f"\tE(pop zero score): {(sum(pop_zero_score)/DEN):.6f}")
#     print(f"\tDiff between E(pop zero score) - E(pop neg score): {(sum(pop_zero_score)/DEN) - (sum(pop_neg_score)/DEN):.6f}")
#     print('\n')
#     print(f"\tE(pop neg rank): {(sum(pop_neg_rank)/DEN):.6f}")
#     print(f"\tE(pop zero rank): {(sum(pop_zero_rank)/DEN):.6f}")
#     print(f"\tDiff between E(pop_neg_rank) - E(pop_zero_rank): {(sum(pop_neg_rank)/DEN) - (sum(pop_zero_rank)/DEN):.6f}")

    print('='*50)
    return

In [22]:
ablation_score_map = map_sorted_preds_to_ranked_items({'ui_scores':ui_scores,'gt':gt, 'golds':golds, 'preds':preds})

22363it [00:01, 19039.84it/s]


In [23]:
for K in [1,2,3,5,10,20]:
    compute_avg_rank_score(ablation_score_map,popItems,K)

K:1
	len(pop_neg_score):  3
	len(pop_zero_score):  3


	E(pop neg score): 1.000000
	E(pop zero score): 1.000000
	Diff between E(pop zero score) - E(pop neg score): 0.000000


	E(pop neg rank): 1.000000
	E(pop zero rank): 1.000000
	Diff between E(pop_neg_rank) - E(pop_zero_rank): 0.000000
K:2
	len(pop_neg_score):  5
	len(pop_zero_score):  5


	E(pop neg score): 1.000000
	E(pop zero score): 1.000000
	Diff between E(pop zero score) - E(pop neg score): 0.000000


	E(pop neg rank): 1.400000
	E(pop zero rank): 1.400000
	Diff between E(pop_neg_rank) - E(pop_zero_rank): 0.000000
K:3
	len(pop_neg_score):  8
	len(pop_zero_score):  8


	E(pop neg score): 0.974924
	E(pop zero score): 1.000000
	Diff between E(pop zero score) - E(pop neg score): 0.025076


	E(pop neg rank): 2.000000
	E(pop zero rank): 1.750000
	Diff between E(pop_neg_rank) - E(pop_zero_rank): 0.250000
K:5
	len(pop_neg_score):  19
	len(pop_zero_score):  19


	E(pop neg score): 0.988408
	E(pop zero score): 1.000000
	Diff between E(pop

In [24]:
for K in [100]:
    compute_avg_rank_score(ablation_score_map,popItems,K)

K:100
	len(pop_neg_score):  22952
	len(pop_zero_score):  22952


	E(pop neg score): 0.170143
	E(pop zero score): 0.708845
	Diff between E(pop zero score) - E(pop neg score): 0.538702


	E(pop neg rank): 64.254836
	E(pop zero rank): 30.244685
	Diff between E(pop_neg_rank) - E(pop_zero_rank): 34.010152


In [25]:
OUTPUT_DIR = os.path.join("top-preds","NEG-ABL")
os.makedirs(OUTPUT_DIR,exist_ok=True)

In [26]:
save_path = os.path.join(OUTPUT_DIR,f"DEEPFM-{DATASET}-preds-rand-1%.pkl")
save_pickle({'ui_scores':ui_scores,
             'gt':gt, 'golds':golds, 
             'preds':preds, 'pop_golds':pop_golds,
             'pop_gt':pop_gt
            },
            save_path)